# 02 · Data Understanding and Cleaning
**Reads:** `data/raw/*.csv`  
**Writes:** `data/processed/flights_clean.parquet`, `outputs/cleaning_log.csv`

In [ ]:
import os, json, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)

from pathlib import Path

# Repo layout (notebooks live in <repo>/notebooks/, everything else is resolved from the repo root):
#   data/raw        <- original Kaggle csv files (flights.csv, airlines.csv, airports.csv)
#   data/processed  <- intermediate files passed from one notebook to the next
#   model/          <- saved models / configs / final bundle
#   outputs/        <- small reports (csv tables)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = ROOT / "data" / "raw"
DATA_PROC = ROOT / "data" / "processed"
MODEL_DIR = ROOT / "model"
OUT_DIR = ROOT / "outputs"
for d in (DATA_RAW, DATA_PROC, MODEL_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEED = 42


In [ ]:
DELAY_MINUTES = 15          # delayed = arrival delay > 15 min
SAMPLE_ROWS = 2_000_000     # random sample after cleaning to keep things fast (set None for ALL rows)

In [ ]:
# Load data (only columns that are known before departure + target)
cols = ["YEAR", "MONTH", "DAY", "DAY_OF_WEEK", "AIRLINE", "FLIGHT_NUMBER", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT",
        "SCHEDULED_DEPARTURE", "SCHEDULED_TIME", "DISTANCE", "SCHEDULED_ARRIVAL", "ARRIVAL_DELAY", "DIVERTED", "CANCELLED"]
df = pd.read_csv(f"{DATA_RAW}/flights.csv", usecols=cols,
                 dtype={"AIRLINE": str, "ORIGIN_AIRPORT": str, "DESTINATION_AIRPORT": str})
airlines = pd.read_csv(f"{DATA_RAW}/airlines.csv")
airports = pd.read_csv(f"{DATA_RAW}/airports.csv")
print(df.shape)
df.head()

## Data understanding

In [ ]:
print(df.describe().T.round(1))
print("\nMissing values:\n", df.isna().sum()[df.isna().sum() > 0])
print("\nCancelled:", df.CANCELLED.sum(), "| Diverted:", df.DIVERTED.sum())

## Cleaning

In [ ]:
log = []
def step(name, before, reason):
    removed = before - len(df)
    log.append({"step": name, "rows_removed": removed, "pct": round(100 * removed / n_start, 2), "reason": reason})

n_start = len(df)

# Cancelled / diverted flights have no arrival delay -> target undefined
b = len(df); df = df[(df.CANCELLED == 0) & (df.DIVERTED == 0)]
step("cancelled / diverted", b, "No arrival delay exists")

# Missing target
b = len(df); df = df.dropna(subset=["ARRIVAL_DELAY"])
step("missing ARRIVAL_DELAY", b, "No label")

# Inconsistent airport format: some rows use numeric ids instead of 3-letter IATA codes
is_iata = df.ORIGIN_AIRPORT.str.fullmatch(r"[A-Z]{3}") & df.DESTINATION_AIRPORT.str.fullmatch(r"[A-Z]{3}")
print("Rows with numeric airport ids:", (~is_iata).sum(), "-> months:", sorted(df.loc[~is_iata, "MONTH"].unique()))
b = len(df); df = df[is_iata]
step("numeric airport ids", b, "Cannot be mapped to IATA codes; keep one consistent format")

# Duplicates
b = len(df)
df = df.drop_duplicates(subset=["MONTH", "DAY", "AIRLINE", "FLIGHT_NUMBER", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "SCHEDULED_DEPARTURE"])
step("duplicates", b, "Same flight recorded twice")

# Codes must exist in the reference tables
b = len(df)
df = df[df.AIRLINE.isin(airlines.IATA_CODE) & df.ORIGIN_AIRPORT.isin(airports.IATA_CODE) & df.DESTINATION_AIRPORT.isin(airports.IATA_CODE)]
step("unknown airline/airport", b, "Not in airlines.csv / airports.csv")

# Invalid values
b = len(df)
bad = (df.SCHEDULED_TIME.isna() | (df.SCHEDULED_TIME <= 0) | (df.DISTANCE <= 0) | (df.SCHEDULED_DEPARTURE > 2359)
       | (df.SCHEDULED_ARRIVAL > 2400) | (df.ORIGIN_AIRPORT == df.DESTINATION_AIRPORT))
df = df[~bad]
step("invalid values", b, "Non-positive time/distance, impossible clock time, origin = destination")

# Outliers: implausible scheduled speed (distance / scheduled time) = data-entry errors
speed = df.DISTANCE / df.SCHEDULED_TIME * 60
b = len(df); df = df[(speed >= 60) & (speed <= 700)]
step("implausible speed", b, "Speed < 60 or > 700 mph")

pd.DataFrame(log)

In [ ]:
# Outliers in ARRIVAL_DELAY (very long delays): KEPT - they are real, and our target is binary (> 15 min) anyway
print(df.ARRIVAL_DELAY.describe(percentiles=[.5, .9, .99]).round(1))
plt.figure(figsize=(8, 3.5))
sns.histplot(df.ARRIVAL_DELAY.clip(-60, 240).sample(min(len(df), 300_000), random_state=SEED), bins=100)
plt.axvline(DELAY_MINUTES, color="red", ls="--"); plt.title("Arrival delay in minutes (clipped for plotting)"); plt.show()

# Target
df["DELAYED"] = (df.ARRIVAL_DELAY > DELAY_MINUTES).astype(int)
print(f"\nShare of delayed flights: {df.DELAYED.mean():.1%}  (imbalanced -> use ROC-AUC / PR-AUC / F1, not accuracy)")

if SAMPLE_ROWS and len(df) > SAMPLE_ROWS:
    df = df.sample(SAMPLE_ROWS, random_state=SEED)
df = df.reset_index(drop=True)
print("Clean data:", df.shape, "| rows removed in total:", n_start - len(df) if not SAMPLE_ROWS else "see log")

## Save cleaned data for the next notebook

In [ ]:
log_df = pd.DataFrame(log)
log_df.to_csv(OUT_DIR / "cleaning_log.csv", index=False)
df.to_parquet(DATA_PROC / "flights_clean.parquet", index=False)      # needs pyarrow
print("Saved:", DATA_PROC / "flights_clean.parquet", df.shape)